# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the [FAIR^2 Croissant](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library. This dataset contains mass spectrometry protein analysis of human-derived extracellular vesicles, including fields for protein accession, description, molecular weight, peptide counts, and calculated protein parameters.

### Dataset Source
The dataset source is defined by a Croissant metadata schema accessible at the URL below.

In [ ]:
# Install mlcroissant if not already installed
!pip install -q mlcroissant

## 1. Data Loading

Load metadata and available record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata

# Display key dataset information
print(f"Dataset Name: {metadata.name}")
print(f"Description:\n{metadata.description}\n")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Version: {getattr(metadata, 'version', None)}")
print(f"License: {getattr(metadata, 'license', None)}")

## 2. Data Overview

Review available record sets and their `@id`s. For each record set, show the fields (`@id`s) and a schema preview.

In [ ]:
# List all record sets and fields by @id
record_sets = ds.record_sets  # This returns a list of mlcroissant.RecordSet objects

print("Available record sets (with @id):")
for rs in record_sets:
    print(f"  - @id: {rs.id} | name: {rs.name}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      - @id: {field.id} | name: {field.name} | dataType: {getattr(field, 'data_type', None)}")

## 3. Data Extraction

Load data from the main record set (typically contains the scientific tabular data) into a DataFrame for analysis.

**All Croissant entity references use their `@id`. In this dataset, the primary tabular record set will be identified as shown above.**

In [ ]:
# List all record set @id's for quick reference
record_set_ids = [rs.id for rs in record_sets]
print("All available record set @id values:")
print(record_set_ids)

# Assuming there is only one main scientific table, select the first record set
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
print(f"Main record set @id: {main_record_set_id}")

# Load all dataframes
dataframes = {}
for rsid in record_set_ids:
    # Each record is a dictionary mapping field @id to value
    records = list(ds.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)

print(f"Columns for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())

# Show first few records
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Here we will perform basic filtering, normalization, and grouping by field using the Croissant `@id` as the DataFrame column key.

First, let's identify a numeric field and a suitable grouping field by inspecting the schema above.

In [ ]:
# Inspect columns for a numeric field to analyze (e.g., "cr:peptide_count"), and a grouping field (e.g., "cr:description", "cr:accession")
df = dataframes[main_record_set_id].copy()

# Example field @id values from the FAIR^2 schema (will need to match actual names):
numeric_field_id = None
group_field_id = None
for col in df.columns:
    # Heuristically pick a numeric field
    if any(x in col.lower() for x in ['count', 'mw', 'cover', 'abundance', 'peptide']):
        if numeric_field_id is None:
            numeric_field_id = col
    # Pick a grouping field
    if any(x in col.lower() for x in ['description', 'accession', 'sample']):
        if group_field_id is None:
            group_field_id = col
print(f"Numeric field selected for analysis: {numeric_field_id}")
print(f"Grouping field selected: {group_field_id}")

# Try to ensure the field is float or int
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Set a threshold for filtering (use the 10th percentile if appropriate)
try:
    threshold = df[numeric_field_id].quantile(0.1)
except Exception:
    threshold = 10

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize the selected field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}':")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

if group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()  # mean value per group
    print(f"\nAverage '{numeric_field_id}' grouped by '{group_field_id}':")
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the chosen numeric field and its means per group (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group if available
if group_field_id in filtered_df.columns:
    top_groups = filtered_df[group_field_id].value_counts().head(10).index  # Top 10 most frequent groups
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=filtered_df[filtered_df[group_field_id].isin(top_groups)], x=group_field_id, y=numeric_field_id)
    plt.xticks(rotation=45, ha='right')
    plt.title(f"{numeric_field_id} by {group_field_id} (top 10 groups)")
    plt.show()

## 6. Conclusion

We successfully explored the FAIR^2 'Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells' dataset using the `mlcroissant` library, referencing entities strictly by their Croissant `@id`. We demonstrated schema inspection, data extraction, simple EDA and basic visuals to facilitate downstream, reproducible research using FAIR record sets.